In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, shutil, json
import torch

from helmholtz_eos_net.test import load_model_from_folder


msize=12
mwidth = 2
fsize=16
alpha=0.3
lsize = 2
framewidth = 3
markeredgewidth=3
markeredgewidth_no=10

print('Imports successful!')

In [ ]:
model_folder = "ray_results/d29fd_00000/"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, config = load_model_from_folder(model_folder, device)


In [ ]:
checkpoint_path = "modified_models/"+model_folder.split("/")[-2]

try:
    os.mkdir(checkpoint_path)
    shutil.copyfile(model_folder+"/config.json", checkpoint_path+"/config.json")
    print("modify config...")
except:
    None

In [ ]:
with open(checkpoint_path+"/config.json") as f:
    config = json.load(f)


In [ ]:
model

In [ ]:
from helmholtz_eos_net.substance_parameters import SubstanceParameters
substance_params = SubstanceParameters(
    csv_path=config["substance_params_csv"],
    id_column=config["id_column"],
    parameter_columns=config["parameter_columns"],
    fixed_columns=config.get("fixed_columns", ["canonical_smiles", "common_name"]),
    embedding_dim=len(config["parameter_columns"]),
)

model.substance_params = substance_params

config["parameter_columns"]

In [ ]:
model

In [ ]:
import torch.nn as nn

#dd = 512
dd = len(config["parameter_columns"])

model.simple_net.bnn[0] = nn.Linear(in_features=dd+2, out_features=2048) 

model.embedding_dim= dd
model.p_dim = dd
model.parameter_columns = config["parameter_columns"] #[ "feature_"+str(i) for i in range(dd) ]

model

In [ ]:
model.simple_net.bnn[0].weight.shape

In [ ]:
checkpoint = checkpoint_path + f"/model_wide_{dd}.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "config": config,
}, checkpoint)


print(f"Saved checkpoint to: {checkpoint}")

In [ ]:
model.substance_params